# Emoji provider classification (IDSI51006 Computer Vision, BSc AIDAMS 2026)

**Team "The Winners"**: Adam Zeraiki, Lina, Charles, Erdem Sanver.
**Task**: given a PNG of an emoji, predict which provider drew it (apple, facebook, google, messenger, mozilla, samsung, whatsapp). 9,879 labelled training images, 9,879 test images, accuracy on Kaggle.

This notebook is the single entry point of the codebase. It runs on **any machine with an NVIDIA GPU** (we used a RunPod RTX 4090 pod; Google Colab and Databricks work the same way) and on a **laptop** for the CPU parts, without edits other than the configuration cell. All logic lives in `src/emoji_provider/` and `scripts/` so that every number in the report can be regenerated from the command line as well (the exact commands of the final runs are in `README.md`).

| Step | What it does | Where it runs |
|---|---|---|
| 1. Setup | find the codebase and the Kaggle archive, install pinned packages | anywhere |
| 2. Data and EDA | native PNG sizes/modes per provider, duplicates, class balance | CPU, ~1 min |
| 3. Preprocessing | alpha-composite on white, keep aspect ratio, pad to a square canvas, cache | CPU, ~1 min |
| 4. Validation protocol | stratified, group-aware split that keeps identical images on one side | CPU |
| 5. Experiment ladder | metadata-only diagnostic, September baseline, compact CNN (+metadata ablation) | CPU, ~1.5 h, or skip |
| 6. Final model | ConvNeXt-Tiny fine-tuned at 224 px, 5 folds, last-epoch checkpoints, flip TTA, soft voting (plus the checkpoint-selection ablation) | **GPU**, 16 min per run on an RTX 4090 |
| 7. Results and submission | out-of-fold metrics, confusion matrix, failure cases, `submission.csv` | CPU |

Smoke mode (`SMOKE = True`) runs every step on a tiny subset in a few minutes to check the installation; it does not produce a competition model.

## 1. Setup

**GPU pod (RunPod or any Jupyter box)**: put the Kaggle archive and the unzipped codebase side by side (we used `/workspace/2-computer-vision-2026-b-sc-aidams-final-proj.zip` and `/workspace/codebase/`), open this notebook from `codebase/notebooks/`, *Run all*. Packages missing from the image (scikit-learn, timm) are installed by the first cell without touching the image's CUDA torch.
**Colab**: upload both zips to `/content` (or mount Drive), choose a GPU runtime, *Run all*.
**Databricks**: upload both zips to a Unity Catalog Volume, import this notebook, attach a GPU cluster (ML runtime), set `CODEBASE_ZIP` and `ZIP_PATH` to the `/Volumes/...` paths, *Run all*; the last cell copies `outputs/` back to the Volume.
**Laptop**: `pip install -r requirements.txt`, put the archive in `data/`, run the notebook or the scripts.

In [ ]:
import os, sys, json, subprocess, zipfile, importlib
from pathlib import Path

ON_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ
ON_COLAB = importlib.util.find_spec("google.colab") is not None
ON_RUNPOD = "RUNPOD_POD_ID" in os.environ or Path("/workspace").is_dir()
print("environment:", "Databricks" if ON_DATABRICKS else "Colab" if ON_COLAB else "RunPod / GPU pod" if ON_RUNPOD else "local")

# ---- where is the codebase? (the folder that contains src/, scripts/, requirements.txt)
CODEBASE_ZIP = None          # e.g. "/Volumes/<catalog>/<schema>/<volume>/emoji_provider_codebase.zip" on Databricks
def locate_codebase() -> Path:
    for cand in [Path.cwd(), Path.cwd().parent, Path("/workspace/codebase"), Path("/workspace/emoji_provider_codebase"),
                 Path("/content/emoji_provider_codebase"), Path("/content")]:
        if (cand / "src" / "emoji_provider").is_dir():
            return cand
    if CODEBASE_ZIP:
        # Databricks: work on the driver's local disk (fast, no Workspace file limits); results are copied to the Volume at the end
        base = Path("/local_disk0") if (ON_DATABRICKS and Path("/local_disk0").is_dir()) else Path("/tmp") if ON_DATABRICKS else Path.cwd()
        target = base / "emoji_provider_codebase"
        with zipfile.ZipFile(CODEBASE_ZIP) as z:
            z.extractall(target)
        inner = next(p for p in [target, *target.iterdir()] if (p / "src" / "emoji_provider").is_dir())
        return inner
    raise FileNotFoundError("codebase not found: set CODEBASE_ZIP or run the notebook from the codebase folder")

ROOT = locate_codebase()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("codebase:", ROOT)

# ---- packages: GPU images (RunPod / Colab / Databricks) keep their own CUDA build of torch; laptops get the exact pins
if ON_DATABRICKS or ON_COLAB or ON_RUNPOD:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "-r", "requirements-gpu-runtime.txt"])
else:
    missing = [l.strip() for l in open("requirements.txt") if l.strip() and not l.startswith("#")]
    try:
        import torch, timm, sklearn, pandas, matplotlib, PIL  # noqa
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])

import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
from IPython.display import display, Image as IPImage
from emoji_provider import data as D, evaluate as E
from emoji_provider.eda import run_eda
print({m: importlib.metadata.version(m) for m in ["torch", "torchvision", "timm", "scikit-learn", "numpy", "pandas", "pillow"]})
DEVICE = D.pick_device(); print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")

## Configuration

Only this cell should need editing. `ZIP_PATH = None` lets the code look in `/workspace`, `/content`, the current folder, `./data` and `~/Downloads`.

In [ ]:
ZIP_PATH   = None            # e.g. "/workspace/2-computer-vision-2026-b-sc-aidams-final-proj.zip" (RunPod) or "/Volumes/<catalog>/<schema>/<volume>/..." (Databricks)
SMOKE      = False                       # True = tiny subset, 1-2 epochs, random init: checks the installation only
OUT_ROOT   = ROOT / ("outputs_smoke" if SMOKE else "outputs")   # everything the run produces goes here (figures, metrics, probabilities, submissions)

RUN_CPU_LADDER = True        # metadata-only diagnostic, September baseline, compact CNN (+ metadata ablation); ~1.5 h on 2 CPU cores, minutes on a GPU
RUN_FINAL_GPU  = True        # ConvNeXt-Tiny 224 px, 5-fold + TTA; needs a GPU (16 min per run on an RTX 4090, about 1 h on a T4)
SELECT_ON      = "last"      # which epoch to keep per fold: "last" (the final model, run B: 97.69% OOF, 0.98319 public) or "val_loss" (run A, 97.29%)
RUN_BOTH_RULES = True        # True = also run the other rule (the ablation of Section 5.2) and write the 10-checkpoint average as a second file
FOLDS, EPOCHS_GPU, SIZE_GPU, BATCH_GPU, LR_GPU = 5, 10, 224, 32, 2e-4
EPOCHS_CPU = 25

# optional post-processing on the final probabilities (both documented in the report, both OFF by default)
USE_DUPLICATE_LOOKUP = False  # test files byte-identical to a training file take that label
USE_CONSISTENCY      = False  # identical test images share one averaged prediction

CHUNKS_DIR = None            # optional: a Volume folder holding the archive split into parts (part_00, part_01, ...); they are joined on local disk
DATA_DIR = ROOT / "data" / "extracted"          # the archive is extracted here once (19,758 PNGs); on Databricks this is the local disk
if CHUNKS_DIR and (ZIP_PATH is None or not Path(ZIP_PATH).is_file()):
    import hashlib
    joined = ROOT / "data" / "2-computer-vision-2026-b-sc-aidams-final-proj.zip"
    joined.parent.mkdir(parents=True, exist_ok=True)
    parts = sorted(Path(CHUNKS_DIR).glob("part_*"))
    assert parts, f"no part_* files in {CHUNKS_DIR}"
    h = hashlib.sha256()
    with open(joined, "wb") as out:
        for part in parts:
            chunk = part.read_bytes(); out.write(chunk); h.update(chunk)
    print(f"joined {len(parts)} parts -> {joined} ({joined.stat().st_size:,} bytes), sha256 {h.hexdigest()}")
    assert h.hexdigest() == "7f8c12a2a36591e45f761b21a49dc2e9be1276e7d04cbf6d06ca8df2d844ad8d", "archive checksum mismatch: a part is missing or corrupt"
    ZIP_PATH = str(joined)
_volume_dir = Path(CHUNKS_DIR).parent if CHUNKS_DIR else (Path(ZIP_PATH).parent if ZIP_PATH else None)
PERSIST_TO = _volume_dir / "emoji_outputs" if (ON_DATABRICKS and _volume_dir) else None   # Volume folder that receives outputs/ at the end
OUT_ROOT.mkdir(parents=True, exist_ok=True)
smoke_flag = ["--smoke"] if SMOKE else []
def run(cmd: list[str]) -> None:
    """Run a script of the codebase and stream its output into the notebook."""
    print("$", " ".join(cmd), flush=True)
    proc = subprocess.Popen([sys.executable, *cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=ROOT)
    for line in proc.stdout:
        if "UserWarning" in line or line.startswith("  warnings.warn"):
            continue
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"{cmd[0]} failed with exit code {proc.returncode}")

## 2. Data and exploratory analysis

The Kaggle archive contains `train/` (9,879 PNGs), `test/` (9,879 PNGs) and `train_labels.csv`. The seven providers are imbalanced: apple, google, samsung, facebook and whatsapp have 1,644 to 1,924 images each, messenger has 580 and mozilla 397, so the majority class alone gives 19.5% accuracy and chance is 14.3%.

Before touching pixels we look at what the **files themselves** say. Each provider exported its emoji with its own pipeline, and that leaves fingerprints: Apple, Google and Samsung files are 72 px wide, WhatsApp 160 px, Facebook, Messenger and Mozilla 240 px; Samsung saved palette (`P`) PNGs where everyone else used `RGBA`; and the RGB values stored *under fully transparent pixels* differ by provider (a PNG encoder leaves whatever was in the buffer). The EDA below measures all of this on the real files, and checks that the test set has the same mix (it does, total variation 0.5%).

In [ ]:
if not (DATA_DIR / "train_labels.csv").is_file():
    D.extract_zip(D.find_zip(ZIP_PATH), DATA_DIR)
cat = D.load_catalogue(DATA_DIR)
print(f"{len(cat.train_ids):,} training / {len(cat.test_ids):,} test images; classes: {cat.class_names}")
display(cat.labels["Label"].value_counts().rename("images").to_frame().T)

EDA_DIR = OUT_ROOT / "eda"
if not (EDA_DIR / "summary.json").is_file():
    summary = run_eda(DATA_DIR, EDA_DIR)
summary = json.loads((EDA_DIR / "summary.json").read_text())
print(json.dumps({k: v for k, v in summary.items() if k not in ("class_counts", "class_share")}, indent=2))
display(pd.read_csv(EDA_DIR / "size_by_provider.csv", index_col=0))
display(pd.read_csv(EDA_DIR / "mode_by_provider.csv", index_col=0))
for f in ["class_distribution.png", "native_size_by_provider.png", "samples_per_provider.png", "alpha_naive_vs_white.png"]:
    display(IPImage(filename=str(EDA_DIR / "figures" / f), width=850))

## 3. Preprocessing

One recipe for every model (function `emoji_provider.data.load_rgb_white`):

1. `convert("RGBA")`, then **alpha-composite on a white background**. The naive `convert("RGB")` used by `skimage.io.imread`/PIL by default *drops* the alpha channel and exposes the provider-specific junk colour hidden under transparent pixels. We want the network to learn the drawing style, not the PNG encoder, so this shortcut is removed on purpose.
2. **Fit inside a square canvas keeping the aspect ratio** (`ImageOps.contain`, Lanczos), then **pad with white**. A few Google files are 72×68, so stretching would distort them.
3. Resize to the model's input size (96 px for the compact CNN, 224 px for ConvNeXt-Tiny), store as `uint8`, cache as `.npy` so reruns do not decode 20k PNGs again.
4. Normalisation: `(x/255 - 0.5)/0.5` for the networks trained from scratch, ImageNet mean/std for the pretrained backbone.

**Augmentation**: horizontal flip and integer translations of up to 4 px (pad with white, random crop). Both are exact pixel operations: no interpolation, so anti-aliased outlines and gradients, which *are* the provider signature, are untouched. Rotation/scale augmentation and label smoothing were tried by Erdem (experiment E7) and cost 1.5 points, which is consistent with this reasoning.

In [ ]:
train_paths = [cat.train_files[i] for i in cat.train_ids]; test_paths = [cat.test_files[i] for i in cat.test_ids]
X96 = D.load_pixels(train_paths, 96, cache=ROOT / "data" / "cache_train_96.npy")
hashes = D.pixel_hashes(X96)
print("duplicates after preprocessing:", D.duplicate_report(hashes, cat.y))
fig, axes = plt.subplots(1, 8, figsize=(13, 2))
for ax, i in zip(axes, np.random.default_rng(1).choice(len(X96), 8, replace=False)):
    ax.imshow(X96[i]); ax.set_title(cat.labels["Label"][i], fontsize=9); ax.axis("off")
plt.show()

## 4. Validation protocol

158 training images are exact duplicates of another training image (same preprocessed pixels; none with conflicting labels). A random split would put copies of the same image on both sides and inflate validation accuracy, so we use **`StratifiedGroupKFold`** with the pixel hash as the group: identical images always land on the same side, and every fold keeps the class proportions.

`data/split_seed42.csv` stores the split used by every single-model experiment (7,900 train / 1,979 validation). It is shipped as a file because the shuffled assignment of `StratifiedGroupKFold` is not stable across scikit-learn versions; all models in the report are scored on exactly these 1,979 images. The 5-fold run of the final model additionally reports **out-of-fold** accuracy on all 9,879 images, and the subset of those that belong to the stored validation set, so the two protocols can be compared directly.

In [ ]:
train_idx, val_idx = D.load_fixed_split(ROOT / "data" / "split_seed42.csv", cat.labels)
print(f"stored split: {len(train_idx):,} train / {len(val_idx):,} validation")
display(pd.DataFrame({"provider": cat.class_names,
                      "train": np.bincount(cat.y[train_idx], minlength=7),
                      "validation": np.bincount(cat.y[val_idx], minlength=7)}).set_index("provider").T)

## 5. Experiment ladder (CPU)

Each rung changes one thing, all scored on the stored validation split:

* **Majority class** and **metadata only** (no pixels: width, height, PIL mode, alpha flag, transparent fraction, optionally the colour under transparency). This measures how much a model could score *without seeing the emoji*.
* **September baseline**: the saved `best_model.pt` (3 conv blocks, no BatchNorm, Adam 1e-3, 20 epochs) re-scored: 76.71%.
* **EmojiCNNv2**: four Conv-BN-ReLU stages (32-64-128-256), global average pooling, dropout, 0.39 M parameters, AdamW + OneCycle, flip/shift augmentation.
* **EmojiCNNv2 + file metadata** (late fusion): the ablation that tells us what the file format adds *on top of* pixels.

The script writes histories, confusion matrices, per-class F1, the most confident mistakes and Grad-CAM maps into `outputs/cpu/`.

In [ ]:
if RUN_CPU_LADDER:
    run(["scripts/run_cpu_experiments.py", "--data-dir", str(DATA_DIR), "--out", str(OUT_ROOT / "cpu"), "--epochs", str(EPOCHS_CPU), *smoke_flag])
    res = json.loads((OUT_ROOT / "cpu" / "results.json").read_text())
    display(pd.DataFrame({k: {"val accuracy": v.get("accuracy"), "macro F1": v.get("macro_f1")} for k, v in res.items() if isinstance(v, dict) and "accuracy" in v}).T.round(4))
    for f in ["per_class_f1_cpu_models.png", "cm_emojicnnv2_norm.png", "gradcam_emojicnnv2.png", "failures_emojicnnv2.png"]:
        display(IPImage(filename=str(OUT_ROOT / "cpu" / "figures" / f), width=850))

## 6. Final model: ConvNeXt-Tiny, fine-tuned at 224 px, 5 folds, flip TTA

**Why this model.** The team's ladder showed that generic ImageNet features are not enough when frozen (ResNet18 frozen: 77.5%), that full fine-tuning is what adapts the filters to rendering style (ResNet18: 93.7%), that ConvNeXt-Tiny's 7×7 depthwise kernels and LayerNorm capture stroke/shading style better (95.9% at 128 px), and that resolution matters most (97.6% at 224 px, the pre-training resolution and close to the 240 px native size of Facebook/Messenger/Mozilla files).

**What this run adds.** (i) 5-fold cross-validation instead of one 80/20 split: five models, each validated on images it never saw, so the reported accuracy is an out-of-fold estimate over all 9,879 images and the ensemble averages five softmax outputs; (ii) horizontal-flip test-time augmentation; (iii) stochastic depth 0.1 and the exact-pixel augmentation above against the train-accuracy-1.000 overfitting seen in E8; (iv) mixed precision.

**Which epoch to keep.** Run A kept, per fold, the epoch with the lowest validation loss (epochs 4 to 8): validation loss bottoms out early while accuracy keeps improving as the OneCycle schedule anneals, a loss/accuracy divergence that is common with cross-entropy once most examples are right and a few confident mistakes dominate the loss. Run B re-ran the same five trainings (identical trajectories, same seeds and GPU) and kept the last epoch: +0.4 points out-of-fold, and it is the final model (0.98319 public). Averaging the ten checkpoints (two per fold) was a tie out-of-fold and scored 0.98279.

Cost: about 2.6 min per fold on an RTX 4090 (16 min per run including inference and figures), 10-12 min per fold on a T4. Set `FOLDS = 1` to train one model on the stored split instead.

In [ ]:
def gpu_out_dir(rule: str) -> Path:
    if SMOKE:
        return OUT_ROOT / ("smoke_gpu" if rule == "val_loss" else "smoke_gpu_last")
    return OUT_ROOT / (f"convnext{SIZE_GPU}" if rule == "val_loss" else f"convnext{SIZE_GPU}_last")   # run A / run B of the report

RULES = [SELECT_ON] + ([r for r in ("val_loss", "last") if r != SELECT_ON] if RUN_BOTH_RULES else [])
GPU_OUT = gpu_out_dir(SELECT_ON)
if RUN_FINAL_GPU:
    if DEVICE.type == "cpu" and not SMOKE:
        print("No GPU detected: this step would take many hours on CPU. Attach a GPU runtime or set SMOKE = True to test the path.")
    else:
        for rule in RULES:
            run(["scripts/train_backbone.py", "--data-dir", str(DATA_DIR), "--out", str(gpu_out_dir(rule)), "--model", "convnext_tiny.fb_in1k",
                 "--size", str(SIZE_GPU), "--epochs", str(EPOCHS_GPU), "--batch", str(BATCH_GPU), "--lr", str(LR_GPU),
                 "--folds", str(FOLDS), "--select-on", rule, *smoke_flag])

## 7. Results and submission

In [ ]:
for rule in RULES:
    d = gpu_out_dir(rule)
    if not (d / "oof_summary.json").is_file():
        continue
    s = json.loads((d / "oof_summary.json").read_text())
    print(f"\n=== {d.name} (select_on={rule}) ===")
    print(f"out-of-fold accuracy over {s['covered_images']:,} images: {s['accuracy']:.4%} ± {s['accuracy_stderr']:.4%} (1 s.e.)")
    print(f"macro F1 {s['macro_f1']:.4f} | balanced accuracy {s['balanced_accuracy']:.4f} | log loss {s['log_loss']:.4f}")
    if "stored_split_validation_oof" in s:
        print("on the stored 1,979-image validation split (out-of-fold):", s["stored_split_validation_oof"])
    display(pd.DataFrame(s["folds"]).round(4))
    display(pd.DataFrame(s["per_class"]).T.round(4))
    for f in ["curves_fold0.png", "cm_oof.png", "cm_oof_norm.png", "failures_oof.png"]:
        if (d / "figures" / f).is_file():
            display(IPImage(filename=str(d / "figures" / f), width=800))

In [ ]:
# Final submission = the SELECT_ON run's own soft vote of its five fold-models (its test_proba_mean.npy).
# With RUN_BOTH_RULES the other run's file is written too, and the 10-checkpoint average as submission_both_rules.csv (ablation).
SUB_DIR = OUT_ROOT / "submissions"; SUB_DIR.mkdir(exist_ok=True)
inputs = [str(GPU_OUT / "test_proba_mean.npy")] if (GPU_OUT / "test_proba_mean.npy").is_file() and not SMOKE else []
if not SMOKE and RUN_BOTH_RULES and all((gpu_out_dir(r) / "test_proba_mean.npy").is_file() for r in RULES):
    run(["scripts/make_final_submission.py", *[str(gpu_out_dir(r) / "test_proba_mean.npy") for r in RULES], "--data-dir", str(DATA_DIR),
         "--out", str(SUB_DIR / "submission_both_rules.csv")])
if not inputs:
    inputs = [str(OUT_ROOT / "cpu" / "test_proba_emojicnnv2.npy")]
rules = (["--duplicate-lookup"] if USE_DUPLICATE_LOOKUP else []) + (["--consistency"] if USE_CONSISTENCY else [])
run(["scripts/make_final_submission.py", *inputs, "--data-dir", str(DATA_DIR), "--out", str(SUB_DIR / "submission.csv"), *rules])
sub = pd.read_csv(SUB_DIR / "submission.csv", dtype=str)
display(sub.head()); print(sub["Label"].value_counts())
if ON_COLAB:
    from google.colab import files; files.download(str(SUB_DIR / "submission.csv"))
print("Submission file:", SUB_DIR / "submission.csv")

### Persist the results

A pod's disk disappears when the pod is terminated and a Databricks driver's disk when the cluster stops. On Databricks this copies `outputs/` next to the archive on the Volume; elsewhere it zips `outputs/` without the 1.1 GB of fold checkpoints (`outputs_results.zip`, next to the codebase) so it can be downloaded from the Jupyter file browser. Download the checkpoints separately if you want to keep them.

In [ ]:
import shutil
if PERSIST_TO is not None:
    PERSIST_TO.mkdir(parents=True, exist_ok=True)
    for sub in sorted(OUT_ROOT.iterdir()):
        dest = PERSIST_TO / sub.name
        if sub.is_dir():
            shutil.copytree(sub, dest, dirs_exist_ok=True)
        else:
            shutil.copy2(sub, dest)
    print("outputs copied to", PERSIST_TO)
    print("submission:", PERSIST_TO / "submissions" / "submission.csv")
else:
    archive = ROOT.parent / "outputs_results.zip"
    with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
        for f in OUT_ROOT.rglob("*"):
            if f.is_file() and f.suffix != ".pt":
                z.write(f, f.relative_to(OUT_ROOT.parent))
    print(f"results stay in {OUT_ROOT}; zipped without checkpoints to {archive} ({archive.stat().st_size / 1e6:.1f} MB)")

## 8. AI usage and reproducibility

* Every AI tool used on this codebase and the prompts are listed in `AI_USAGE.md` (the course asks for full transparency). The teacher also allowed AI for writing the paper; its use is recorded there too.
* `requirements.txt` pins the versions used for the CPU experiments; `requirements-gpu-runtime.txt` lists what a GPU image needs on top of its own CUDA `torch` (scikit-learn, timm). Versions actually used are printed in Setup and stored in every `*_run.json` (field `versions`); for the 17 October runs they are in `outputs/runpod_training_logs.txt` (torch 2.8.0+cu128 on an RTX 4090).
* Seeds: 42 everywhere (`data.seed_everything`), `cudnn.deterministic = True`. GPU kernels can still introduce tiny run-to-run differences; the stored split, fold table (`folds.csv`) and saved probabilities make every reported number re-checkable without retraining.